In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

# ============================================================
# 1. Load SCD2 Work Table
# ============================================================

scd2_work_df = spark.table(
    "customer360_dev.control.customer_scd2_work"
)

target_table = "customer360_dev.gold.dim_customer"

dim_customer_delta = DeltaTable.forName(
    spark,
    target_table
)

In [0]:
type2_changes_df = (
    scd2_work_df
    .filter(
        F.col("scd2_action") == "TYPE2_CHANGE"
    )
)

type2_change_count = type2_changes_df.count()

print(
    "TYPE2 changes to expire:",
    type2_change_count
)

TYPE2 changes to expire: 0


In [0]:
type2_changes_df = (
    type2_changes_df
    .withColumn(
        "change_timestamp",
        F.coalesce(
            F.col("source_updated_at"),
            F.current_timestamp()
        )
    )
)

In [0]:
(
    dim_customer_delta.alias("tgt")

    .merge(
        type2_changes_df.alias("src"),
        """
        tgt.customer_id = src.customer_id
        AND tgt.is_current = true
        """
    )

    .whenMatchedUpdate(
        set={
            "effective_to":
                "src.change_timestamp",

            "is_current":
                "false",

            "source_updated_at":
                "current_timestamp()"
        }
    )

    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
still_current_count = (
    spark.table(target_table)
    .alias("tgt")
    .join(
        type2_changes_df
        .select("customer_id")
        .distinct()
        .alias("src"),

        F.col("tgt.customer_id")
        ==
        F.col("src.customer_id"),

        "inner"
    )
    .filter(
        F.col("tgt.is_current") == True
    )
    .count()
)

print(
    "TYPE2 customers still current after expiry:",
    still_current_count
)

TYPE2 customers still current after expiry: 0


In [0]:
display(
    spark.table(target_table)
    .filter(
        F.col("is_current") == False
    )
    .orderBy(
        F.col("source_updated_at").desc()
    )
    .select(
        "customer_sk",
        "customer_id",
        "loyalty_tier",
        "customer_status",
        "effective_from",
        "effective_to",
        "is_current"
    )
    .limit(50)
)

customer_sk,customer_id,loyalty_tier,customer_status,effective_from,effective_to,is_current
